In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import informal_jobs_model as ijm

/Users/gperaza/Research/informal-jobs-model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Both surveys are loaded through the project's data packages instead of local CSV files:

1. **ENOE** via [`mxcensus`](https://github.com/CentroFuturoCiudades/mxcensus): `load_enoe_persons` returns the SDEM roster joined with the two occupation questionnaires (COE1, COE2) for each quarter in `ijm.ENOE_PERIODS` (2022 T1 – 2023 T4, pooled; the reference quarter `ijm.ENOE_PERIOD` = 2023 T1 matches the OD fieldwork), restricted to Jalisco (`ent == 14`). Pooling eight quarters gives ~50k workers instead of ~7k; survey weights are divided by the number of quarters so weighted totals remain the average quarterly population, and the panel structure (a dwelling is visited up to five times) is handled downstream by grouping on the cross-quarter household key (`ijm.ENOE_GROUP_KEYS`). `load_enoe(table="sdem")` provides the full dwelling roster used to count household size.
2. **Origin–Destination survey** via [`eodgdl`](https://github.com/CentroFuturoCiudades/eodgdl): `load_eod` returns dwellings (`viv`), persons (`hab`), trips and legs with snake_case column names; persons are joined with the dwelling attributes (municipality, AGEB, centrality, household size).

On first use each package downloads and caches the raw tables (`~/Library/Caches/mxcensus`, `~/Library/Caches/eodgdl`); set `MXCENSUS_CACHE_DIR` / `EODGDL_CACHE_DIR` to relocate the caches.

Stage-1 constants (pooled quarters, state, identifier keys, output columns, renames, the OD employment categories and DENUE release) are configuration, not code: `src/informal_jobs_model/config/enoe.yaml` and `src/informal_jobs_model/config/od.yaml`, loaded with `ijm.load_config`. ENOE workers are INEGI's employed population (`clase2 == 1`, i.e. worked last week or had a job and was temporarily absent) on the analytical universe: definitive interview (`r_def == 0`), habitual or new residents (`c_res in {1, 3}`) and ages 12–98 (`min_age`/`max_age` in `enoe.yaml`). INEGI publishes employment for ages 15+; the floor is lowered to 12 because the OD survey records working 12–14 year olds.

In [2]:
enoe = ijm.generate_enoe_dataframe()  # ijm.ENOE_PERIODS pooled; weights divided by their number so totals stay at population scale
print("ENOE workers per quarter:", enoe["period"].value_counts().sort_index().to_dict())
od = ijm.generate_od_dataframe()

ENOE workers per quarter: {'2022t1': 6456, '2022t2': 6436, '2022t3': 6286, '2022t4': 6256, '2023t1': 6973, '2023t2': 6503, '2023t3': 6459, '2023t4': 6338}


For the ENOE, employed persons in Jalisco are selected, household size is computed from the full SDEM roster of each dwelling, and the employment variables and weighting factors needed later are retained (`ijm.ENOE_OUTPUT_COLUMNS`). All columns are integer codes.

For the OD, the persons table is joined with the dwelling table (municipality, AGEB, centrality and household size) and filtered to employed persons. Raw OD columns whose names collide with the harmonized attributes created in the next stage carry a `_raw` suffix (`ijm.OD_RAW_COLUMN_RENAMES`).

In [3]:
print(f"ENOE workers: {enoe.shape}")
print(f"OD workers: {od.shape}")

assert enoe["ent"].eq(ijm.ENOE_STATE_CODE).all(), (
    "ENOE contains observations outside Jalisco."
)
assert enoe["survey_weight"].notna().all(), "ENOE contains missing survey weights."
assert od["expansion_factor"].notna().all(), "OD contains missing expansion factors."

display(enoe.head())
display(od.head())

ENOE workers: (51707, 33)
OD workers: (26913, 74)


,period,tipo,mes_cal,cd_a,ent,con,v_sel,n_hog,h_mud,n_ren,...,dwelling_size,p4,p4b,p4e,p4f,p4h,hogar_trabajadores,hogar_ninos_6_11,tue2,seg_soc
0,2022t1,1,96,2,14,40001,1,1,0,1,...,6,1,2,<NA>,<NA>,<NA>,4,0,4,1
1,2022t1,1,96,2,14,40001,1,1,0,2,...,6,1,4,<NA>,<NA>,1,4,0,1,1
2,2022t1,1,96,2,14,40001,1,1,0,4,...,6,1,4,3,<NA>,<NA>,4,0,2,2
3,2022t1,1,96,2,14,40001,1,1,0,6,...,6,1,4,<NA>,<NA>,1,4,0,1,1
4,2022t1,1,96,2,14,40001,2,1,1,1,...,4,1,4,2,<NA>,<NA>,4,0,2,2


,folio_vivienda,folio_habitante,fecha,salio_casa_ayer,razon_no_viaje,viajes_contados,dia_semana_viajes,parentesco_raw,sexo_nacimiento,genero_identidad,...,centralidad,dwelling_size,n_autos_camionetas,destino_ambito,dest_establecimientos_log,dest_share_grandes,dest_share_comercio,dest_share_gobierno_otro_agricultura,dest_share_manufactura_construccion,dest_share_servicios_transporte
0,1,1,2023-01-25 00:00:00+00:00,Sí,<NA>,7,Jueves,Jefe del hogar,Hombres,Hombres,...,30F,5,2,ageb_urbana,5.099866,0.092025,0.447853,0.0000,0.159509,0.392638
1,2,1,2023-01-26 00:00:00+00:00,Sí,<NA>,2,Miércoles,Jefe del hogar,Hombres,Hombres,...,09,3,0,ageb_urbana,5.438079,0.192140,0.279476,0.0131,0.030568,0.676856
2,2,2,2023-01-26 00:00:00+00:00,No,Otros (especifique),0,<NA>,Otro parentesco,Hombres,Hombres,...,09,3,0,desconocido,NaN,NaN,NaN,NaN,NaN,NaN
3,2,3,2023-01-26 00:00:00+00:00,No,Otros (especifique),0,<NA>,Compañero,Hombres,Hombres,...,09,3,0,desconocido,NaN,NaN,NaN,NaN,NaN,NaN
4,3,1,2023-01-26 00:00:00+00:00,No,Otros (especifique),0,<NA>,Cónyuge,Mujeres,Mujeres,...,09,3,0,desconocido,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
output_directory = ROOT / "outputs"
output_directory.mkdir(exist_ok=True)

enoe.to_parquet(output_directory / "enoe_workers.parquet", index=False)
od.to_parquet(output_directory / "od_workers.parquet", index=False)